<a href="https://colab.research.google.com/github/EfiDefiyati/Modul-Praktikum-Deep-Learning/blob/main/M01_123450005.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modul 1: Fondasi Jaringan Saraf, FNN, Aktivasi, dan Loss

**Nama:** Efi Defiyati
**NIM:** 123450005  
**Kelas:** RC  
**Tanggal:** 2026-09-15  

**Berkas pengumpulan:** `M01_NIM.ipynb`, `M01_NIM.pdf`, dan `M01_NIM_metrics.csv`.

> Seluruh kode, eksperimen, grafik, dan analisis merupakan pekerjaan individual. Beri atribusi pada kode yang diadaptasi dari sumber lain.

## Petunjuk

1. Ganti seluruh penanda `TODO`.
2. Jangan mengubah protokol eksperimen kecuali diminta.
3. Gunakan validation set untuk memilih model. Test set hanya untuk model final.
4. Sebelum mengumpulkan, jalankan **Restart Kernel and Run All**.
5. Pertahankan semua hasil eksperimen, termasuk hasil yang tidak sesuai hipotesis.

In [ ]:
pip install torch numpy pandas scikit-learn matplotlib

In [ ]:
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.datasets import make_moons
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

NIM = '123450005'
assert NIM.isdigit() and len(NIM) >= 4, 'Isi NIM dengan angka sebelum melanjutkan.'
NIM_LAST4 = int(NIM[-4:])
SEED = 1000 + NIM_LAST4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)

environment = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'sklearn': sklearn.__version__,
    'torch': torch.__version__,
    'device': str(DEVICE),
    'seed': SEED,
}

environment

## A. Pre-lab - 10 poin

Jawab sebelum sesi praktikum.

1. **Parameter vs hyperparameter:** Parameter adalah nilai yang dipelajari oleh model selama proses training, seperti bobot dan bias. Sedangkan hyperparameter adalah nilai yang ditentukan sebelum training, seperti learning rate, batch size, jumlah epoch, hidden size, dan jenis fungsi aktivasi.
2. **Mengapa tumpukan layer affine tanpa aktivasi dapat diciutkan:** Tumpukan layer affine tanpa fungsi aktivasi dapat digabung menjadi satu layer affine karena komposisi dari beberapa operasi linear atau affine tetap menghasilkan transformasi affine. Oleh karena itu, tanpa fungsi aktivasi non-linear, penambahan layer tidak memberikan kemampuan untuk mempelajari pola yang lebih kompleks.
3. **Shape bobot layer 2 masukan -> 8 keluaran:** Bobot layer dengan 2 masukan dan 8 keluaran memiliki shape (8, 2), sedangkan bias-nya memiliki shape (8,).
4. **Mengapa `BCEWithLogitsLoss` menerima logit:** BCEWithLogitsLoss menerima logit karena fungsi tersebut sudah menggabungkan sigmoid dengan binary cross-entropy sehingga perhitungannya lebih stabil secara numerik. Oleh karena itu, sigmoid tidak perlu ditambahkan pada output model.

### Hipotesis awal

Prediksi aktivasi dan hidden size yang akan memberi validation loss terbaik. Jelaskan alasannya sebelum menjalankan eksperimen.

**Jawaban:** Saya memperkirakan ReLU dengan hidden size 16 akan menghasilkan validation loss terbaik karena ReLU dapat membantu model mempelajari pola non-linear pada dataset make_moons dan hidden size yang lebih besar memberikan kapasitas model yang lebih tinggi.

## B. Neuron dan fungsi aktivasi dengan NumPy - 20 poin

In [ ]:
def sigmoid_np(z: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z)

    positive = z >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-z[positive]))

    exp_z = np.exp(z[~positive])
    out[~positive] = exp_z / (1.0 + exp_z)

    return out


def relu_np(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z)
    return np.maximum(0, z)


def leaky_relu_np(z: np.ndarray, alpha: float = 0.01) -> np.ndarray:
    z = np.asarray(z)
    return np.where(z >= 0, z, alpha * z)


def neuron_batch(X: np.ndarray, w: np.ndarray, b: float, activation):
    X = np.asarray(X)
    w = np.asarray(w)

    if X.ndim != 2:
        raise ValueError("X harus berdimensi 2: (n_samples, n_features).")

    if w.ndim != 1:
        raise ValueError("w harus berdimensi 1.")

    if X.shape[1] != w.shape[0]:
        raise ValueError(
            f"Shape tidak cocok: X memiliki {X.shape[1]} fitur, "
            f"sedangkan w memiliki {w.shape[0]} elemen."
        )

    z = X @ w + b
    a = activation(z)

    return z, a

In [ ]:
grid = np.linspace(-6, 6, 600)

activation_values = {
    'Sigmoid': sigmoid_np(grid),
    'Tanh': np.tanh(grid),
    'ReLU': relu_np(grid),
    'Leaky ReLU': leaky_relu_np(grid),
}

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

for ax, (name, values) in zip(axes.ravel(), activation_values.items()):
    ax.plot(grid, values, linewidth=2, label=name)
    ax.set_title(name)
    ax.set_xlabel('z')
    ax.set_ylabel('aktivasi(z)')
    ax.grid(alpha=0.3)
    ax.legend()

fig.suptitle('Perbandingan Fungsi Aktivasi')
fig.tight_layout()
plt.show()

### Interpretasi aktivasi

- Aktivasi yang berpusat di nol: Fungsi aktivasi yang berpusat di nol adalah tanh karena nilai output-nya berada pada rentang -1 sampai 1.
- Aktivasi yang tepat nol untuk masukan negatif: ReLU menghasilkan nilai tepat nol ketika masukannya bernilai negatif.
- Aktivasi yang mudah jenuh: Sigmoid dan tanh mudah mengalami saturasi ketika nilai input sangat besar atau sangat kecil sehingga perubahan output menjadi sangat kecil.
- Mengapa aktivasi linear tidak cukup untuk `make_moons`: Aktivasi linear tidak cukup untuk dataset make_moons karena pola datanya tidak dapat dipisahkan dengan garis lurus. Fungsi aktivasi non-linear diperlukan agar jaringan dapat membentuk decision boundary yang melengkung.

## C. Forward pass manual - bagian dari 20 poin pemetaan

Gunakan arsitektur $2 \rightarrow 2 \rightarrow 1$, ReLU pada hidden layer, dan target $y=1$. Jangan mengubah bobot.

In [ ]:
x_manual = np.array([[2.0, -1.0]], dtype=np.float32)
W1 = np.array([[0.5, -0.5], [1.0, 1.0]], dtype=np.float32)
b1 = np.array([0.0, 0.0], dtype=np.float32)
W2 = np.array([[2.0, -1.0]], dtype=np.float32)
b2 = np.array([0.5], dtype=np.float32)
y_manual = np.array([[1.0]], dtype=np.float32)

# Forward pass
z1_np = x_manual @ W1.T + b1
h_np = np.maximum(0, z1_np)

logit_np = h_np @ W2.T + b2
prob_np = sigmoid_np(logit_np)

# BCE untuk y = 1
bce_np = -(
    y_manual * np.log(prob_np)
    + (1 - y_manual) * np.log(1 - prob_np)
)

parameter_count_manual = (
    W1.size + b1.size +
    W2.size + b2.size
)

print("z1 =", z1_np)
print("h =", h_np)
print("logit =", logit_np)
print("probabilitas =", prob_np)
print("BCE =", bce_np)
print("Jumlah parameter =", parameter_count_manual)

### Tabel shape dan nilai

| Tensor | Shape | Nilai |
|---|---:|---:|
| $x$ | (1, 2) | [[2.0, -1.0]] |
| $W^{(1)}$ | (2, 2) | tersedia pada kode |
| $z^{(1)}$ | (1, 2) | [[1.5, 1.0]] |
| $h$ | (1, 2) | [[1.5, 1.0]] |
| logit | (1, 1) | [[2.5]] |
| probabilitas | 	(1, 1) | [[0.9241]] |
| BCE | skalar | 0.0789
 |

**Jumlah parameter dan perhitungannya:** Jumlah parameter pada jaringan tersebut adalah 9, yaitu 4 bobot dan 2 bias pada layer pertama serta 2 bobot dan 1 bias pada layer kedua.

Hasil forward pass:
Nilai hidden layer diperoleh dari hasil perkalian input dengan bobot ditambah bias, kemudian dilewatkan ke fungsi ReLU. Dari perhitungan tersebut diperoleh hidden layer sebesar [1.5, 1.0], logit sebesar 2.5, probabilitas sekitar 0.9241, dan nilai BCE sekitar 0.0789.

## D. Pencocokan NumPy dan PyTorch - 20 poin

Salin bobot ke `nn.Sequential`. Jangan menambahkan sigmoid ke model karena loss menerima logit.

In [ ]:
manual_model = nn.Sequential(
    nn.Linear(2, 2),
    nn.ReLU(),
    nn.Linear(2, 1),
)

with torch.no_grad():
    manual_model[0].weight.copy_(torch.from_numpy(W1))
    manual_model[0].bias.copy_(torch.from_numpy(b1))
    manual_model[2].weight.copy_(torch.from_numpy(W2))
    manual_model[2].bias.copy_(torch.from_numpy(b2))

manual_model = manual_model.to(DEVICE)

x_manual_t = torch.from_numpy(x_manual).to(DEVICE)
y_manual_t = torch.from_numpy(y_manual).to(DEVICE)

with torch.no_grad():
    torch_logit = manual_model(x_manual_t)
    torch_probability = torch.sigmoid(torch_logit)
    torch_loss = nn.functional.binary_cross_entropy_with_logits(
        torch_logit,
        y_manual_t
    )

print("PyTorch logit:", torch_logit.item())
print("PyTorch probability:", torch_probability.item())
print("PyTorch BCE:", torch_loss.item())

assert np.max(
    np.abs(torch_logit.detach().cpu().numpy() - logit_np)
) < 1e-6

assert abs(
    torch_loss.item() - float(bce_np)
) < 1e-6

assert (
    sum(p.numel() for p in manual_model.parameters())
    == parameter_count_manual
)

print("Pemeriksaan NumPy-PyTorch: LULUS")

### Checkpoint menit ke-85

Tunjukkan kepada asisten:

- plot empat fungsi aktivasi;
- tabel shape dan parameter; dan
- selisih forward NumPy-PyTorch kurang dari $10^{-6}$.

**Status/verifikasi asisten:** TODO

## E. Dataset dan protokol eksperimen

Kode split dan standardisasi diberikan agar fokus tetap pada FNN. Jangan menggunakan test set sampai model final dipilih.

In [ ]:
X, y = make_moons(n_samples=600, noise=0.22, random_state=SEED)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)
y_train_f = y_train.astype(np.float32).reshape(-1, 1)
y_val_f = y_val.astype(np.float32).reshape(-1, 1)
y_test_f = y_test.astype(np.float32).reshape(-1, 1)

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'n': [len(y_train), len(y_val), len(y_test)],
    'positive_rate': [y_train.mean(), y_val.mean(), y_test.mean()],
})
display(split_summary)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(X_train_s[:, 0], X_train_s[:, 1], c=y_train, cmap='coolwarm', s=24, alpha=0.75)
ax.set(title='Train Set make_moons', xlabel='fitur 1', ylabel='fitur 2')
ax.grid(alpha=0.2)
plt.show()

### Pemeriksaan anti-leakage

Jelaskan mengapa scaler hanya di-fit pada train set dan bukan seluruh data.

**Jawaban:** TODO

## F. Model dan utilitas training

Lengkapi pemilihan aktivasi dan model. Training loop diberikan; mekanismenya dibahas pada Modul 2.

In [ ]:
def activation_layer(name: str) -> nn.Module:
    name = name.lower()

    if name == 'relu':
        return nn.ReLU()
    elif name == 'tanh':
        return nn.Tanh()
    elif name == 'sigmoid':
        return nn.Sigmoid()
    else:
        raise ValueError(
            f"Aktivasi tidak dikenal: {name}. "
            "Gunakan 'relu', 'tanh', atau 'sigmoid'."
        )


def build_model(hidden_dim: int, activation: str, seed: int = SEED):
    seed_everything(seed)

    model = nn.Sequential(
        nn.Linear(2, hidden_dim),
        activation_layer(activation),
        nn.Linear(hidden_dim, 1),
    )

    return model.to(DEVICE)


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


model_check = build_model(8, 'relu')

assert count_parameters(model_check) == 33

print("Jumlah parameter 2 -> 8 -> 1:", count_parameters(model_check))

In [ ]:
def as_tensor(array):
    return torch.as_tensor(array, dtype=torch.float32)

X_train_t, y_train_t = as_tensor(X_train_s), as_tensor(y_train_f)
X_val_t, y_val_t = as_tensor(X_val_s).to(DEVICE), as_tensor(y_val_f).to(DEVICE)
X_test_t, y_test_t = as_tensor(X_test_s).to(DEVICE), as_tensor(y_test_f).to(DEVICE)

def evaluate(model, X_tensor, y_tensor):
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, y_tensor).item()
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).to(torch.int64)
        accuracy = (predictions == y_tensor.to(torch.int64)).float().mean().item()
    return {
        'loss': loss,
        'accuracy': accuracy,
        'probabilities': probabilities.cpu().numpy().ravel(),
        'predictions': predictions.cpu().numpy().ravel(),
    }

def train_model(model, epochs=200, learning_rate=0.05, batch_size=32, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size, shuffle=True, generator=generator,
    )
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    criterion = nn.BCEWithLogitsLoss()
    history = []
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(X_batch)
        val_metrics = evaluate(model, X_val_t, y_val_t)
        history.append({
            'epoch': epoch,
            'train_loss': loss_sum / len(X_train_t),
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
        })
    return pd.DataFrame(history), time.perf_counter() - start

## G. Baseline dan latihan kelas

Baseline: ReLU, hidden size 8, SGD, learning rate 0.05, batch size 32, dan 200 epoch.

Latihan individual: digit terakhir NIM 0-4 memakai tanh; 5-9 memakai sigmoid. Prediksi hasil sebelum menjalankan.

**Prediksi:** TODO

In [ ]:
# TODO: latih baseline, tampilkan kurva train/validation loss, dan catat metrik.
# TODO: latih satu variasi individual dengan komponen lain tetap.
raise NotImplementedError('Selesaikan baseline dan latihan individual.')

In [ ]:
baseline_model = build_model(8, 'relu')

baseline_history, baseline_time = train_model(
    baseline_model,
    epochs=200,
    learning_rate=0.05,
    batch_size=32,
    seed=SEED
)

baseline_train = evaluate(baseline_model, X_train_t.to(DEVICE), y_train_t.to(DEVICE))
baseline_val = evaluate(baseline_model, X_val_t, y_val_t)

print("Baseline")
print("Train loss:", baseline_train['loss'])
print("Train accuracy:", baseline_train['accuracy'])
print("Validation loss:", baseline_val['loss'])
print("Validation accuracy:", baseline_val['accuracy'])
print("Training time:", baseline_time)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(
    baseline_history['epoch'],
    baseline_history['train_loss'],
    label='Train loss'
)
ax.plot(
    baseline_history['epoch'],
    baseline_history['val_loss'],
    label='Validation loss'
)
ax.set_title('Baseline: ReLU, Hidden Size 8')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
individual_model = build_model(8, 'sigmoid')

individual_history, individual_time = train_model(
    individual_model,
    epochs=200,
    learning_rate=0.05,
    batch_size=32,
    seed=SEED
)

individual_train = evaluate(
    individual_model,
    X_train_t.to(DEVICE),
    y_train_t.to(DEVICE)
)

individual_val = evaluate(
    individual_model,
    X_val_t,
    y_val_t
)

print("Individual: Sigmoid, hidden size 8")
print("Train loss:", individual_train['loss'])
print("Train accuracy:", individual_train['accuracy'])
print("Validation loss:", individual_val['loss'])
print("Validation accuracy:", individual_val['accuracy'])
print("Training time:", individual_time)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(
    individual_history['epoch'],
    individual_history['train_loss'],
    label='Train loss'
)
ax.plot(
    individual_history['epoch'],
    individual_history['val_loss'],
    label='Validation loss'
)
ax.set_title('Individual: Sigmoid, Hidden Size 8')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## H. Tugas individual: enam eksperimen - 20 poin

Jalankan kombinasi aktivasi `{relu, tanh, sigmoid}` dan hidden size `{4, 16}`. Gunakan validation loss untuk memilih model.

In [ ]:
ACTIVATIONS = ['relu', 'tanh', 'sigmoid']
HIDDEN_SIZES = [4, 16]
EPOCHS = 200
LEARNING_RATE = 0.05
BATCH_SIZE = 32

experiment_rows = []
trained_models = {}
histories = {}

for activation in ACTIVATIONS:
    for hidden_dim in HIDDEN_SIZES:

        run_id = f"{activation}_h{hidden_dim}"

        model = build_model(
            hidden_dim=hidden_dim,
            activation=activation,
            seed=SEED
        )

        history, train_time = train_model(
            model,
            epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
            batch_size=BATCH_SIZE,
            seed=SEED
        )

        train_metrics = evaluate(
            model,
            X_train_t.to(DEVICE),
            y_train_t.to(DEVICE)
        )

        val_metrics = evaluate(
            model,
            X_val_t,
            y_val_t
        )

        experiment_rows.append({
            'run_id': run_id,
            'activation': activation,
            'hidden_dim': hidden_dim,
            'parameters': count_parameters(model),
            'epochs': EPOCHS,
            'learning_rate': LEARNING_RATE,
            'batch_size': BATCH_SIZE,
            'train_loss': train_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'],
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'train_time_sec': train_time,
        })

        trained_models[run_id] = model
        histories[run_id] = history


results = (
    pd.DataFrame(experiment_rows)
    .sort_values('val_loss')
    .reset_index(drop=True)
)

assert len(results) == 6

display(results)

results.to_csv(
    f'M01_{NIM_LAST4:04d}_metrics.csv',
    index=False
)

print(f"CSV tersimpan sebagai M01_{NIM_LAST4:04d}_metrics.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

labels = results['run_id']
values = results['val_loss']

bars = ax.bar(labels, values)

ax.set_title('Validation Loss Enam Eksperimen')
ax.set_xlabel('Konfigurasi')
ax.set_ylabel('Validation BCE Loss')
ax.grid(axis='y', alpha=0.25)

for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{value:.4f}',
        ha='center',
        va='bottom'
    )

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## I. Evaluasi model final - 15 poin

Pilih satu model dengan validation loss terendah. Baru setelah itu evaluasi test set satu kali.

In [ ]:
best_run_id = results.loc[0, 'run_id']
final_model = trained_models[best_run_id]

print("Model final:", best_run_id)
print(results.iloc[0])


In [ ]:
final_test = evaluate(
    final_model,
    X_test_t,
    y_test_t
)

print("Final model:", best_run_id)
print("Test loss:", final_test['loss'])
print("Test accuracy:", final_test['accuracy'])


In [ ]:
cm = confusion_matrix(
    y_test.astype(int),
    final_test['predictions']
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[0, 1]
)

fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix - {best_run_id}')
plt.show()


In [ ]:
def plot_decision_boundary(model, X_values, y_values, title):
    x_min, x_max = X_values[:, 0].min() - 0.5, X_values[:, 0].max() + 0.5
    y_min, y_max = X_values[:, 1].min() - 0.5, X_values[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid_points = np.c_[
        xx.ravel(),
        yy.ravel()
    ].astype(np.float32)

    grid_tensor = torch.from_numpy(grid_points).to(DEVICE)

    model.eval()

    with torch.no_grad():
        logits = model(grid_tensor)
        probabilities = torch.sigmoid(logits).cpu().numpy().reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7, 5.5))

    contour = ax.contourf(
        xx,
        yy,
        probabilities,
        levels=50,
        cmap='RdBu',
        alpha=0.55
    )

    ax.contour(
        xx,
        yy,
        probabilities,
        levels=[0.5],
        colors='black',
        linewidths=2
    )

    scatter = ax.scatter(
        X_values[:, 0],
        X_values[:, 1],
        c=y_values,
        cmap='coolwarm',
        edgecolor='k',
        s=30,
        alpha=0.85
    )

    ax.set_title(title)
    ax.set_xlabel('Fitur 1')
    ax.set_ylabel('Fitur 2')
    ax.grid(alpha=0.2)

    fig.colorbar(
        contour,
        ax=ax,
        label='Probabilitas kelas 1'
    )

    plt.tight_layout()
    plt.show()


plot_decision_boundary(
    final_model,
    X_test_s,
    y_test,
    f'Decision Boundary Model Final: {best_run_id}'
)

## J. Analisis dan refleksi - 10 poin

Jawab dengan merujuk angka atau grafik.

1. Apakah hidden size lebih besar selalu memperbaiki validation loss? **TODO**
2. Aktivasi mana yang paling stabil pada dua hidden size? **TODO**
3. Adakah konfigurasi dengan accuracy serupa tetapi BCE berbeda? Mengapa? **TODO**
4. Di bagian mana decision boundary paling tidak pasti? **TODO**
5. Apa satu keterbatasan eksperimen ini? **TODO**
6. Apakah hasil sesuai hipotesis awal? **TODO**

## Checklist pengumpulan - 5 poin reproduksibilitas

- [ ] Identitas, seed, versi library, dan device tercantum.
- [ ] Forward NumPy-PyTorch memiliki selisih kurang dari $10^{-6}$.
- [ ] Enam eksperimen tercatat pada notebook dan CSV.
- [ ] Test set hanya dipakai untuk model final.
- [ ] Semua grafik memiliki judul, label sumbu, dan legenda/caption.
- [ ] Notebook lolos Restart Kernel and Run All.
- [ ] Tidak ada path absolut atau data pribadi.
- [ ] Sumber eksternal telah diberi atribusi.

### Pernyataan orisinalitas

Saya menyatakan bahwa kode, eksperimen, visualisasi, dan analisis pada notebook ini merupakan pekerjaan individual saya.

**Nama dan tanggal:** TODO